In [37]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [38]:
X,y = make_classification(
    n_samples = 1000,
    n_features = 20,
    n_informative = 10,
    n_redundant = 5,
    n_classes = 2,
    weights = [0.9,0.1],
    random_state = 42 
)

In [39]:
X_train,X_test,y_train,y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [40]:
scaler = StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaler = scaler.transform(X_test)

In [41]:
# Phase 2: The Baseline & The "Metric Trap"
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,f1_score

model = RandomForestClassifier(random_state=42)
model.fit(X_train_scaled,y_train)

y_pred = model.predict(X_test_scaler)

acc = accuracy_score(y_test,y_pred)
f1Score = f1_score(y_test,y_pred)
print("Accuracy :",acc*100,"%")
print(f"F1 Score : {f1Score*100:.2f} %")

Accuracy : 91.0 %
F1 Score : 30.77 %


In [42]:
from sklearn.model_selection import GridSearchCV
parameters = {
    "n_estimators": [50, 100, 200], 
    "max_depth": [None, 10, 20], 
    "min_samples_split": [2, 5, 10]
}

gridsearch = GridSearchCV(
    RandomForestClassifier(random_state=42),
    parameters,
    scoring = "accuracy",
    cv = 5
)

gridsearch.fit(X_train_scaled, y_train)

print(f"Best Parameters (Accuracy): {gridsearch.best_params_}")

Best Parameters (Accuracy): {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 50}


In [43]:
gridsearch = GridSearchCV(
    RandomForestClassifier(random_state=42),
    parameters,
    scoring = "f1",
    cv = 5
)

gridsearch.fit(X_train_scaled, y_train)

print(f"Best Parameters (F1): {gridsearch.best_params_}")

Best Parameters (F1): {'max_depth': None, 'min_samples_split': 10, 'n_estimators': 50}


In [44]:
import time
from sklearn.model_selection import GridSearchCV

start = time.time()

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    parameters,
    scoring = "f1",
    cv = 5
)
grid.fit(X_train_scaled,y_train)
grid_time = time.time() - start
grid_best_f1 = grid.best_score_

print(f"Grid Time : {grid_time:.2f}")
print(f"Grid Best F1 : {grid_best_f1:.2f}")

Grid Time : 64.06
Grid Best F1 : 0.40


In [45]:
from sklearn.model_selection import RandomizedSearchCV
params = {
    "n_estimators": list(range(10, 501)),   
    "max_depth": [None, 10, 20, 30, 40],
    "min_samples_split": [2, 5, 10, 15]
}
start = time.time()

random = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    params,
    n_iter=20,
    scoring="f1",
    cv = 5,
    random_state=42
)
random.fit(X_train_scaled,y_train)

random_time = time.time() - start
random_best_f1 = random.best_score_

print(f"Random Time : {random_time:.2f}")
print(f"Random Best F1 : {random_best_f1:.2f}")

Random Time : 127.98
Random Best F1 : 0.41


In [46]:
import pandas as pd
results = pd.DataFrame(
    {
        "Method" : ["GridSearch", "RandomSearch"],
        "Time Taken (Sec)" : [grid_time , random_time],
        "Best F1 Score" : [grid_best_f1, random_best_f1]
    }
)
print(results)

         Method  Time Taken (Sec)  Best F1 Score
0    GridSearch         64.062566       0.403463
1  RandomSearch        127.979955       0.411172
